# 🎯 DS-CNN Keyword Spotting — обучение + квантизация на Google Colab

**Что делает этот ноутбук:**
1. Монтирует Google Drive для персистентного хранения (датасет + модели переживают рестарт runtime)
2. Загружает проект `nn/` из GitHub или zip
3. Скачивает Google Speech Commands v2
4. Обучает FP32 DS-CNN (~10 мин на T4)
5. PTQ INT8 квантизация
6. QAT INT8 квантизация
7. Сравнительный анализ + экспорт в C-массив
8. Скачивает готовые артефакты на локальную машину

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**

---

## 0️⃣ Настройки

Здесь единственная ячейка, которую нужно отредактировать под себя:

In [ ]:
# =====================================================================
# НАСТРОЙКИ — отредактируй под себя
# =====================================================================

# Вариант загрузки проекта: "git" или "zip"
# - "git"  — клонировать из GitHub (repo + branch ниже)
# - "zip"  — загрузить zip вручную через Files panel
SOURCE_MODE = "git"  # @param ["git", "zip"]

# Если git:
GIT_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
GIT_BRANCH = "main"  # @param {type:"string"}
# Подпапка с nn/ внутри репо (пусто если nn/ в корне)
NN_SUBDIR = "nn"  # @param {type:"string"}

# Google Drive: куда сохранять датасет и результаты
DRIVE_PROJECT_DIR = "kws_ds_cnn"  # @param {type:"string"}

# Симлинкить датасет на Drive (чтобы не качать 2.3 GB каждый раз)
USE_DRIVE_DATASET = True  # @param {type:"boolean"}

# Симлинкить results/ на Drive (модели + графики переживают рестарт)
USE_DRIVE_RESULTS = True  # @param {type:"boolean"}

print("✅ Настройки заданы")

## 1️⃣ Проверка GPU + монтирование Google Drive

In [ ]:
# Проверка GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU найден: {gpus[0].name}")
    # Разрешаем динамическое выделение памяти (чтобы не забрать всю VRAM сразу)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n⚠️ GPU НЕ найден! Runtime → Change runtime type → T4 GPU")
    print("Обучение на CPU займёт ~45 мин вместо ~10 на T4")

In [ ]:
# Монтирование Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = Path(f"/content/drive/MyDrive/{DRIVE_PROJECT_DIR}")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"✅ Drive смонтирован, проект: {DRIVE_BASE}")

## 2️⃣ Загрузка проекта nn/

In [ ]:
import shutil

WORK_DIR = Path("/content/nn")

if SOURCE_MODE == "git":
    repo_dir = Path("/content/_repo")
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    !git clone --depth 1 --branch {GIT_BRANCH} {GIT_REPO} /content/_repo
    src = repo_dir / NN_SUBDIR if NN_SUBDIR else repo_dir
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(src, WORK_DIR)
    shutil.rmtree(repo_dir)
    print(f"✅ Проект склонирован из {GIT_REPO} ({GIT_BRANCH}) → {WORK_DIR}")

elif SOURCE_MODE == "zip":
    from google.colab import files as colab_files
    print("Загрузите zip-архив с папкой nn/ внутри:")
    uploaded = colab_files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -o "{zip_name}" -d /content/_unzipped
    # Ищем nn/ внутри распакованного
    candidates = list(Path("/content/_unzipped").rglob("config.py"))
    if not candidates:
        raise FileNotFoundError("Не нашёл config.py внутри архива")
    nn_root = candidates[0].parent
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(nn_root, WORK_DIR)
    shutil.rmtree(Path("/content/_unzipped"))
    print(f"✅ Проект распакован в {WORK_DIR}")

# Проверяем структуру
required = ["config.py", "train.py", "models/ds_cnn.py", "data/dataset.py"]
for f in required:
    assert (WORK_DIR / f).exists(), f"Не найден {f} в {WORK_DIR}"
print("✅ Структура проекта ок:")
!find {WORK_DIR} -name '*.py' | head -20

## 3️⃣ Установка зависимостей

In [ ]:
# Colab уже имеет TF предустановленный, но нам нужны конкретные версии.
# Сначала проверяем текущую версию TF в Colab:
import tensorflow as tf
print(f"Текущая версия TF в Colab: {tf.__version__}")

# Если Colab имеет TF 2.15/2.16+ — можно попробовать на нём.
# Если QAT сломается, раскомментируй строку ниже:
# !pip install tensorflow==2.14.1

!pip install -q tensorflow-model-optimization==0.8.0 \
                 librosa==0.10.1 soundfile==0.12.1 \
                 scikit-learn==1.3.2 seaborn==0.13.1 \
                 tqdm pandas matplotlib

print("\n✅ Зависимости установлены")
!python -c "import tensorflow as tf; print(f'TF {tf.__version__}, GPU: {tf.config.list_physical_devices(\"GPU\")}')"
!python -c "import tensorflow_model_optimization as tfmot; print(f'tfmot {tfmot.__version__}')"

## 4️⃣ Симлинки на Google Drive

Датасет (~2.3 GB) и результаты (модели, графики) сохраняются на Drive.
При рестарте runtime не нужно перекачивать и переобучать.

In [ ]:
import os
from pathlib import Path

WORK_DIR = Path("/content/nn")
DRIVE_BASE = Path(f"/content/drive/MyDrive/{DRIVE_PROJECT_DIR}")

def symlink_to_drive(local_rel: str, drive_subdir: str):
    """Создаёт папку на Drive и симлинк local → Drive."""
    local = WORK_DIR / local_rel
    target = DRIVE_BASE / drive_subdir
    target.mkdir(parents=True, exist_ok=True)

    if local.is_symlink():
        local.unlink()
    elif local.exists():
        # Если папка уже существует — переносим содержимое на Drive
        import shutil
        for item in local.iterdir():
            dst = target / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(local)

    local.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(target), str(local))
    print(f"  🔗 {local_rel}/ → {target}")

if USE_DRIVE_DATASET:
    symlink_to_drive("data/speech_commands_v0.02", "dataset/speech_commands_v0.02")
    # Архив тоже на Drive (чтобы не качать повторно)
    archive_drive = DRIVE_BASE / "dataset" / "speech_commands_v0.02.tar.gz"
    archive_local = WORK_DIR / "data" / "speech_commands_v0.02.tar.gz"
    if archive_drive.exists() and not archive_local.exists():
        os.symlink(str(archive_drive), str(archive_local))
        print(f"  🔗 архив симлинкнут с Drive")

if USE_DRIVE_RESULTS:
    symlink_to_drive("results", "results")

print("\n✅ Симлинки готовы")

## 5️⃣ Скачивание датасета

~2.3 GB. Если датасет уже на Drive — пропустится автоматически.

In [ ]:
%cd /content/nn
!python -m data.download

In [ ]:
# Если датасет на Drive и архив ещё локальный — перенести на Drive тоже
if USE_DRIVE_DATASET:
    archive_local_real = WORK_DIR / "data" / "speech_commands_v0.02.tar.gz"
    archive_drive = DRIVE_BASE / "dataset" / "speech_commands_v0.02.tar.gz"
    if archive_local_real.exists() and not archive_local_real.is_symlink() and not archive_drive.exists():
        import shutil
        shutil.move(str(archive_local_real), str(archive_drive))
        os.symlink(str(archive_drive), str(archive_local_real))
        print("Архив перенесён на Drive")
    else:
        print("Архив уже на Drive или не требует переноса")

## 6️⃣ Генерация манифестов train/val/test

In [ ]:
%cd /content/nn
!python -m data.preprocess

In [ ]:
# Быстрая проверка манифестов
import pandas as pd
for name in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/manifests/{name}.csv')
    print(f"{name:>5}: {len(df):>6} samples | classes: {df.label.nunique()} | distribution:")
    print(f"        {dict(df.label.value_counts().head(5))}...\n")

## 7️⃣ Обучение FP32 baseline

~10 мин на T4 (30 эпох, batch 100).
Модель сохраняется в `results/models/`.

In [ ]:
%cd /content/nn

import time
t0 = time.time()

!python train.py

elapsed = time.time() - t0
print(f"\n⏱️ Обучение заняло {elapsed/60:.1f} мин")

In [ ]:
# Посмотреть confusion matrix
from IPython.display import Image, display
cm_path = WORK_DIR / "results" / "plots" / "cm_fp32.png"
if cm_path.exists():
    display(Image(filename=str(cm_path), width=600))
else:
    print(f"Нет файла {cm_path}")

In [ ]:
# Кривые обучения из CSV-лога
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(WORK_DIR / "results" / "logs" / "train.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(log['epoch'], log['loss'], label='train')
ax1.plot(log['epoch'], log['val_loss'], label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax1.grid(alpha=0.3)

ax2.plot(log['epoch'], log['acc'] * 100, label='train')
ax2.plot(log['epoch'], log['val_acc'] * 100, label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy, %'); ax2.legend(); ax2.set_title('Accuracy')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(WORK_DIR / 'results' / 'plots' / 'training_curves.png'), dpi=150)
plt.show()

print(f"\nФинальная val_acc: {log['val_acc'].iloc[-1]*100:.2f}%")
print(f"Лучшая val_acc:    {log['val_acc'].max()*100:.2f}% (epoch {log['val_acc'].idxmax()})")

## 8️⃣ PTQ INT8 квантизация

~1-2 мин.

In [ ]:
%cd /content/nn
!python quantize_ptq.py

In [ ]:
from IPython.display import Image, display
cm = WORK_DIR / 'results' / 'plots' / 'cm_ptq.png'
if cm.exists():
    display(Image(filename=str(cm), width=600))

## 9️⃣ QAT INT8 квантизация

10 эпох fine-tune, ~5 мин на T4.

In [ ]:
%cd /content/nn

t0 = time.time()
!python quantize_qat.py
print(f"\n⏱️ QAT занял {(time.time()-t0)/60:.1f} мин")

In [ ]:
from IPython.display import Image, display
cm = WORK_DIR / 'results' / 'plots' / 'cm_qat.png'
if cm.exists():
    display(Image(filename=str(cm), width=600))

## 📊 Сравнительный анализ

In [ ]:
%cd /content/nn
!python compare_models.py

In [ ]:
# Показать accuracy vs size
from IPython.display import Image, display
p = WORK_DIR / 'results' / 'plots' / 'accuracy_vs_size.png'
if p.exists():
    display(Image(filename=str(p), width=500))

# Маркдаун-таблица
from IPython.display import Markdown
md = WORK_DIR / 'results' / 'comparison.md'
if md.exists():
    display(Markdown(md.read_text()))

## 📦 Экспорт в C-массив для ESP32

In [ ]:
%cd /content/nn

# Экспортируем QAT модель (она обычно лучше PTQ)
!python export_to_c.py \
    --input results/models/ds_cnn_qat_int8.tflite \
    --output results/c_export

print("\nСодержимое results/c_export/:")
!ls -lh results/c_export/

## 📥 Скачать артефакты на локальную машину

Всё уже на Google Drive в папке `kws_ds_cnn/`, но можно скачать самое важное zip-ом прямо из Colab:

In [ ]:
import shutil
from pathlib import Path

export_dir = Path("/content/export_bundle")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir()

# Копируем самое важное
files_to_export = [
    # TFLite модели
    ("results/models/ds_cnn_ptq_int8.tflite", "models/ds_cnn_ptq_int8.tflite"),
    ("results/models/ds_cnn_qat_int8.tflite", "models/ds_cnn_qat_int8.tflite"),
    # C-массив
    ("results/c_export/model_data.cc", "c_export/model_data.cc"),
    ("results/c_export/model_data.h", "c_export/model_data.h"),
    # Анализ
    ("results/comparison.md", "comparison.md"),
    # Графики
    ("results/plots/accuracy_vs_size.png", "plots/accuracy_vs_size.png"),
    ("results/plots/cm_fp32.png", "plots/cm_fp32.png"),
    ("results/plots/cm_ptq.png", "plots/cm_ptq.png"),
    ("results/plots/cm_qat.png", "plots/cm_qat.png"),
    ("results/plots/training_curves.png", "plots/training_curves.png"),
    # Логи
    ("results/logs/train.csv", "logs/train.csv"),
    ("results/logs/qat.csv", "logs/qat.csv"),
]

for src_rel, dst_rel in files_to_export:
    src = WORK_DIR / src_rel
    dst = export_dir / dst_rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"  ✅ {dst_rel} ({src.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"  ⚠️  {src_rel} не найден, пропускаю")

# Собираем zip
zip_path = shutil.make_archive("/content/kws_results", "zip", export_dir)
print(f"\n📦 {zip_path} ({Path(zip_path).stat().st_size / 1024:.1f} KB)")

In [ ]:
# Скачать через браузер
from google.colab import files as colab_files
colab_files.download("/content/kws_results.zip")

## 🛠️ Дополнительно: эксперименты с гиперпараметрами

Если нужно попробовать другие параметры — правь `config.py` прямо в Colab:

In [ ]:
# Пример: поменять эпохи и переобучить
# Раскомментируй и отредактируй:

# import importlib, sys
# sys.path.insert(0, str(WORK_DIR))
#
# # Патчим config на лету
# import config
# config.EPOCHS = 50
# config.BATCH_SIZE = 64
# config.DS_CNN_CONFIG['num_ds_blocks'] = 5  # попробовать больше блоков
# config.DS_CNN_CONFIG['ds_filters'] = 128
#
# # Переименовать выходные файлы
# config.FP32_H5 = config.MODEL_DIR / 'ds_cnn_fp32_v2.h5'
# config.FP32_SAVEDMODEL = config.MODEL_DIR / 'ds_cnn_fp32_v2_saved_model'
#
# # Запуск
# !cd /content/nn && python train.py

## ❓ Восстановление после рестарта runtime

Если Colab отключился или рестартнулся, запусти эту ячейку —
она переподключит Drive, переустановит депы и восстановит симлинки.
После этого можно продолжить с любого шага (PTQ, QAT, compare, export).

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ ПОСЛЕ РЕСТАРТА
# ============================================================
# Настройки (продублированы из ячейки 0)
SOURCE_MODE = "git"
GIT_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
GIT_BRANCH = "main"
NN_SUBDIR = "nn"
DRIVE_PROJECT_DIR = "kws_ds_cnn"
USE_DRIVE_DATASET = True
USE_DRIVE_RESULTS = True

# --- 1. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- 2. Deps ---
!pip install -q tensorflow-model-optimization==0.8.0 \
                 librosa==0.10.1 soundfile==0.12.1 \
                 scikit-learn==1.3.2 seaborn==0.13.1 \
                 tqdm pandas matplotlib

# --- 3. Проект ---
import shutil, os
from pathlib import Path

WORK_DIR = Path("/content/nn")
DRIVE_BASE = Path(f"/content/drive/MyDrive/{DRIVE_PROJECT_DIR}")

if not WORK_DIR.exists():
    if SOURCE_MODE == "git":
        !git clone --depth 1 --branch {GIT_BRANCH} {GIT_REPO} /content/_repo
        src = Path("/content/_repo") / NN_SUBDIR if NN_SUBDIR else Path("/content/_repo")
        shutil.copytree(src, WORK_DIR)
        shutil.rmtree(Path("/content/_repo"))
    print(f"✅ Проект восстановлен: {WORK_DIR}")

# --- 4. Симлинки ---
def symlink_to_drive(local_rel, drive_subdir):
    local = WORK_DIR / local_rel
    target = DRIVE_BASE / drive_subdir
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        for item in local.iterdir():
            dst = target / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(target), str(local))
    print(f"  🔗 {local_rel} → {target}")

if USE_DRIVE_DATASET:
    symlink_to_drive("data/speech_commands_v0.02", "dataset/speech_commands_v0.02")
    archive_drive = DRIVE_BASE / "dataset" / "speech_commands_v0.02.tar.gz"
    archive_local = WORK_DIR / "data" / "speech_commands_v0.02.tar.gz"
    if archive_drive.exists() and not archive_local.exists():
        os.symlink(str(archive_drive), str(archive_local))

if USE_DRIVE_RESULTS:
    symlink_to_drive("results", "results")

# --- 5. Проверка ---
import tensorflow as tf
print(f"\n✅ TF {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")
print(f"✅ Модели на Drive: {list((DRIVE_BASE / 'results' / 'models').glob('*')) if (DRIVE_BASE / 'results' / 'models').exists() else 'none yet'}")
print(f"✅ Можно продолжать с любого шага")

---

## 📝 Заметки

- **Проект на Drive:** после первого запуска всё лежит в `My Drive/kws_ds_cnn/`
- **Датасет:** ~2.5 GB на Drive, не качается повторно
- **Модели:** .h5, .tflite, SavedModel — всё на Drive в `results/models/`
- **Графики:** confusion matrices, training curves — в `results/plots/`
- **Для ESP32:** скачиваете `kws_results.zip` → копируете `c_export/model_data.{cc,h}` в `esp32_firmware/components/nn_inference/src/`
- **TensorBoard:** `%load_ext tensorboard\n%tensorboard --logdir /content/nn/results/logs/tensorboard`